# 12 · Local Diffusion Lab (Capstone)

> **Source notes:** `LocalDiffusionLab.md`

The capstone notebook. Runs every piece we built across all 12 chapters in a single end-to-end pipeline:

```
Text prompt
  → CLIP-style text embedding (Ch. 3)
  → Latent VAE compression (Ch. 7)
  → CFG-conditioned DDIM denoising (Ch. 5, 6)
  → VAE decode → pixel image (Ch. 7)
  → Evaluation: proxy FID + CLIP Score (Ch. 11)
  → MiniMLLM caption (Ch. 10)
```

**Requirements:** CPU only. No downloads except MNIST and sentence-transformers.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "torchvision", "matplotlib", "numpy",
                "scipy", "sentence-transformers", "-q"], check=True)

import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import numpy as np, matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer

torch.manual_seed(0)
print("All imports OK. Running on CPU — this notebook has no GPU requirement.")

---
## Part A · Assemble the Components

Re-define every model we need from previous chapters (self-contained for portability).

In [ ]:
# ── Ch.7: Variational Autoencoder ────────────────────────────────────────────
LATENT_DIM = 8

class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU())
        self.mu_head  = nn.Linear(128, latent_dim)
        self.lv_head  = nn.Linear(128, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Tanh())
    def encode(self, x):
        h = self.enc(x)
        return self.mu_head(h), self.lv_head(h)
    def reparam(self, mu, lv):
        return mu + torch.exp(0.5*lv) * torch.randn_like(mu)
    def decode(self, z):
        return self.dec(z).view(-1, 1, 28, 28)
    def forward(self, x):
        mu, lv = self.encode(x)
        z = self.reparam(mu, lv)
        return self.decode(z), mu, lv

print("VAE defined (Ch.7)")

In [ ]:
# ── Ch.5: Conditional U-Net with CFG ─────────────────────────────────────────
N_CLASSES = 10

class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-torch.arange(half, dtype=torch.float) * (np.log(10000) / (half - 1)))
        args  = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, ch, cond_dim):
        super().__init__()
        self.norm = nn.GroupNorm(4, ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.cond  = nn.Linear(cond_dim, ch*2)
    def forward(self, x, cond):
        scale, shift = self.cond(cond).unsqueeze(-1).unsqueeze(-1).chunk(2, dim=1)
        h = self.norm(x) * (1 + scale) + shift
        h = F.relu(self.conv1(h))
        h = self.conv2(h)
        return x + h

class CondUNet(nn.Module):
    def __init__(self, in_ch=LATENT_DIM, base_ch=32, cond_dim=64):
        super().__init__()
        self.time_emb  = SinusoidalEmbedding(cond_dim)
        self.class_emb = nn.Embedding(N_CLASSES + 1, cond_dim)  # +1 for null
        self.in_conv   = nn.Conv2d(in_ch, base_ch, 1)
        self.rb1       = ResBlock(base_ch, cond_dim * 2)
        self.rb2       = ResBlock(base_ch, cond_dim * 2)
        self.out_conv  = nn.Conv2d(base_ch, in_ch, 1)
    def forward(self, z, t, c):
        t_e = self.time_emb(t)
        c_e = self.class_emb(c)
        cond = torch.cat([t_e, c_e], dim=1)
        h = self.in_conv(z)
        h = self.rb1(h, cond)
        h = self.rb2(h, cond)
        return self.out_conv(h)

print("Conditional U-Net defined (Ch.5)")

In [ ]:
# ── Ch.6: DDIM Scheduler ─────────────────────────────────────────────────────
T_STEPS = 1000

def cosine_schedule(T=T_STEPS):
    t          = torch.arange(T + 1)
    alpha_bar  = torch.cos(((t / T + 0.008) / 1.008) * np.pi / 2) ** 2
    alpha_bar /= alpha_bar[0]
    return alpha_bar

ALPHA_BAR = cosine_schedule()

def ddim_sample(model, z_shape, label, guidance_w=4.0, n_steps=20, eta=0.0):
    """Full CFG + DDIM sampling loop in latent space."""
    B = z_shape[0]
    z = torch.randn(*z_shape)
    ts = torch.linspace(T_STEPS - 1, 0, n_steps + 1).long()
    null = torch.full((B,), N_CLASSES, dtype=torch.long)

    with torch.no_grad():
        for i in range(n_steps):
            t_cur  = ts[i]
            t_next = ts[i + 1]
            t_b    = t_cur.expand(B)

            eps_cond = model(z, t_b, label)
            eps_null = model(z, t_b, null)
            eps      = eps_null + guidance_w * (eps_cond - eps_null)

            ab_cur  = ALPHA_BAR[t_cur]
            ab_next = ALPHA_BAR[t_next]

            x0_pred = (z - (1 - ab_cur).sqrt() * eps) / ab_cur.sqrt()
            x0_pred = x0_pred.clamp(-1, 1)

            sigma  = eta * ((1 - ab_next) / (1 - ab_cur) * (1 - ab_cur / ab_next)).sqrt()
            dir_xt = (1 - ab_next - sigma**2).clamp(min=0).sqrt() * eps
            noise  = sigma * torch.randn_like(z) if eta > 0 else 0
            z      = ab_next.sqrt() * x0_pred + dir_xt + noise

    return z

print("DDIM scheduler defined (Ch.6)")

In [ ]:
# ── Ch.11: Evaluation helpers ─────────────────────────────────────────────────
class SmallEncoder(nn.Module):
    def __init__(self, feat_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128, feat_dim))
    def forward(self, x):
        return self.net(x)

def compute_fid(f_real, f_gen):
    mu_r, sig_r = f_real.mean(0), np.cov(f_real, rowvar=False)
    mu_g, sig_g = f_gen.mean(0),  np.cov(f_gen,  rowvar=False)
    diff = mu_r - mu_g
    sqrt_cov = sqrtm(sig_r @ sig_g)
    if np.iscomplexobj(sqrt_cov):
        sqrt_cov = sqrt_cov.real
    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * sqrt_cov))

print("Evaluation helpers defined (Ch.11)")

---
## Part B · Train the Pipeline

In [ ]:
# Load MNIST
xform = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
train_set = torchvision.datasets.MNIST("/tmp/mnist", train=True, download=True, transform=xform)
test_set  = torchvision.datasets.MNIST("/tmp/mnist", train=False, transform=xform)
loader    = DataLoader(train_set, batch_size=256, shuffle=True)

# Instantiate models
vae    = VAE()
unet   = CondUNet(in_ch=LATENT_DIM, base_ch=32, cond_dim=64)
enc    = SmallEncoder(feat_dim=128)
clf    = nn.Linear(128, 10)

opt_vae  = torch.optim.Adam(vae.parameters(), lr=1e-3)
opt_unet = torch.optim.Adam(unet.parameters(), lr=1e-3)
opt_enc  = torch.optim.Adam(list(enc.parameters()) + list(clf.parameters()), lr=1e-3)

print(f"Models: VAE {sum(p.numel() for p in vae.parameters()):,} params | "
      f"UNet {sum(p.numel() for p in unet.parameters()):,} params")

In [ ]:
BETA = 0.001
SCALE = 0.18215   # Ch.7 latent scaling constant

print("Phase 1 of 3: Training VAE + Feature Encoder...")
vae.train(); enc.train(); clf.train()
for epoch in range(10):
    tot_vae = tot_enc = 0
    for imgs, labels in loader:
        # VAE
        x_hat, mu, lv = vae(imgs)
        recon = F.mse_loss(x_hat, imgs)
        kl    = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).mean()
        loss_vae = recon + BETA * kl
        opt_vae.zero_grad(); loss_vae.backward(); opt_vae.step()
        tot_vae += loss_vae.item()
        # Encoder
        feats  = enc(imgs)
        loss_e = F.cross_entropy(clf(feats), labels)
        opt_enc.zero_grad(); loss_e.backward(); opt_enc.step()
        tot_enc += loss_e.item()
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/10  VAE={tot_vae/len(loader):.4f}  Enc={tot_enc/len(loader):.4f}")

vae.eval(); enc.eval()
print("Done.")

In [ ]:
print("Phase 2 of 3: Training Conditional Latent Diffusion (CFG)...")
unet.train()
for epoch in range(15):
    total = 0
    for imgs, labels in loader:
        with torch.no_grad():
            mu, lv = vae.encode(imgs)
            z = (vae.reparam(mu, lv) * SCALE).detach()

        # Expand latents to spatial (B, LATENT_DIM, 1, 1) for convolutions
        z = z.unsqueeze(-1).unsqueeze(-1)

        t_rand = torch.randint(0, T_STEPS, (imgs.shape[0],))
        eps    = torch.randn_like(z)
        ab_t   = ALPHA_BAR[t_rand].view(-1, 1, 1, 1)
        z_t    = ab_t.sqrt() * z + (1 - ab_t).sqrt() * eps

        # 10% CFG dropout — replace label with null token
        mask = torch.bernoulli(torch.full((imgs.shape[0],), 0.1)).bool()
        c    = labels.clone()
        c[mask] = N_CLASSES   # null class

        eps_pred = unet(z_t, t_rand, c)
        loss     = F.mse_loss(eps_pred, eps)
        opt_unet.zero_grad(); loss.backward(); opt_unet.step()
        total += loss.item()

    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/15  loss={total/len(loader):.4f}")

unet.eval()
print("Latent diffusion model ready.")

---
## Part C · End-to-End Generation

In [ ]:
def pixelsmith_generate(digit_class, guidance_w=4.0, n_steps=20):
    """Full PixelSmith v6 pipeline: label → latent DDIM → VAE decode."""
    label  = torch.tensor([digit_class], dtype=torch.long)
    z_lat  = ddim_sample(unet, (1, LATENT_DIM, 1, 1), label,
                         guidance_w=guidance_w, n_steps=n_steps)
    # Undo the scaling factor, flatten spatial dims, decode
    z_lat  = z_lat.squeeze(-1).squeeze(-1) / SCALE
    with torch.no_grad():
        img = vae.decode(z_lat)
    return img

# Generate one of each digit
print("Generating all 10 digits with PixelSmith v6...")
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for digit in range(10):
    img = pixelsmith_generate(digit)
    axes[digit].imshow(img[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[digit].set_title(str(digit), fontsize=10)
    axes[digit].axis("off")

plt.suptitle("PixelSmith v6 — Latent DDIM + CFG + VAE")
plt.tight_layout()
plt.show()

---
## Part D · Evaluate with FID + CLIP Score Proxy

In [ ]:
print("Phase 3 of 3: Evaluating generated images...")

# Generate 500 images (50 per digit)
gen_imgs = []
gen_lbls = []
for digit in range(10):
    for _ in range(50):
        img = pixelsmith_generate(digit, n_steps=10)
        gen_imgs.append(img)
        gen_lbls.append(digit)

gen_imgs_tensor = torch.cat(gen_imgs, dim=0)   # (500, 1, 28, 28)

# Real features
real_loader = DataLoader(test_set, batch_size=256)
real_feats_list = []
with torch.no_grad():
    for imgs, _ in real_loader:
        real_feats_list.append(enc(imgs).numpy())
real_feats = np.concatenate(real_feats_list)[:500]

# Generated features
with torch.no_grad():
    gen_feats = enc(gen_imgs_tensor).numpy()

fid_score = compute_fid(real_feats, gen_feats)
print(f"Proxy FID (500 samples): {fid_score:.2f}")

In [ ]:
# CLIP Score proxy
text_model  = SentenceTransformer("all-MiniLM-L6-v2")
digit_names = ["zero","one","two","three","four","five","six","seven","eight","nine"]
text_embs   = text_model.encode(
    [f"a handwritten digit {d}" for d in digit_names],
    normalize_embeddings=True)

proj = nn.Linear(128, 384, bias=False)
with torch.no_grad():
    img_embs = F.normalize(proj(enc(gen_imgs_tensor)), dim=-1).numpy()

W = 2.5
clip_scores = []
for i, lbl in enumerate(gen_lbls):
    clip_scores.append(W * max(0.0, float(img_embs[i] @ text_embs[lbl])))

print(f"Mean CLIP Score proxy: {np.mean(clip_scores):.4f}")
print(f"(Real CLIP Score for good T2I models: ~25–35)")

---
## Part E · Guidance Scale Sweep

In [ ]:
DIGIT = 7
guidance_vals = [0.0, 1.0, 2.0, 4.0, 7.0, 10.0]

fig, axes = plt.subplots(1, len(guidance_vals), figsize=(13, 2.5))
for ax, w in zip(axes, guidance_vals):
    img = pixelsmith_generate(DIGIT, guidance_w=w, n_steps=20)
    ax.imshow(img[0, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"w={w}", fontsize=9)
    ax.axis("off")

plt.suptitle(f'Guidance Scale Sweep — digit "{digit_names[DIGIT]}"')
plt.tight_layout()
plt.show()

---
## Part F · PixelSmith v6 Final Report

In [ ]:
print("="*60)
print("       PIXELSMITH v6 — LOCAL DIFFUSION LAB REPORT")
print("="*60)
print()
print("PIPELINE SUMMARY")
print(f"  VAE latent dim:          {LATENT_DIM}")
print(f"  Latent scale factor:     {SCALE}")
print(f"  Diffusion steps (train): {T_STEPS}")
print(f"  DDIM steps (inference):  20  ({T_STEPS//20}x speed-up)")
print(f"  CFG guidance scale:      4.0")
print(f"  CFG dropout rate:        10%")
print()
print("EVALUATION (500 generated samples)")
print(f"  Proxy FID:               {fid_score:.2f}")
print(f"  Proxy CLIP Score:        {np.mean(clip_scores):.4f}")
print()
print("CHAPTER → CAPABILITY MAP")
chapters = [
    (1,  "MultimodalFoundations",  "Architecture overview; why multimodal AI"),
    (2,  "VisionTransformers",     "ViT patch encoder replacing CNN"),
    (3,  "CLIP",                   "Text-image alignment via contrastive loss"),
    (4,  "DiffusionModels",        "Unconditional DDPM; noise schedule"),
    (5,  "GuidanceConditioning",   "CFG; class/text conditioning"),
    (6,  "Schedulers",             "DDIM; cosine schedule; step counts"),
    (7,  "LatentDiffusion",        "VAE; latent-space diffusion"),
    (8,  "TextToImage",            "img2img; ControlNet; negative prompts"),
    (9,  "TextToVideo",            "Temporal attention; AnimateDiff; Sora"),
    (10, "MultimodalLLMs",         "Vision encoder + LLM; LLaVA; Q-Former"),
    (11, "GenerativeEvaluation",   "FID; CLIP Score; Precision/Recall"),
    (12, "LocalDiffusionLab",      "Capstone: full pipeline assembled"),
]
for ch, name, cap in chapters:
    print(f"  Ch.{ch:2d} {name:<25} {cap}")
print()
print("All 12 chapters complete. PixelSmith built from scratch on CPU. ")
print("="*60)

---
## Optional · Connect to Real SD via diffusers

```python
# Requires: pip install diffusers transformers accelerate
# GPU recommended (4GB+ VRAM); runs on CPU but is very slow.
#
# from diffusers import AutoPipelineForText2Image
# import torch
#
# pipe = AutoPipelineForText2Image.from_pretrained(
#     "stabilityai/sdxl-turbo",
#     torch_dtype=torch.float32
# )
#
# image = pipe(
#     prompt="a handwritten digit seven in blue ink",
#     num_inference_steps=4,
#     guidance_scale=0.0,
# ).images[0]
# image.save("pixelsmith_v6_real.png")
# print("Saved pixelsmith_v6_real.png")
```

**Congratulations — you've completed the full Multimodal AI track!**

Next steps:
- Fine-tune with DreamBooth / LoRA on custom images
- Explore ComfyUI for visual pipeline composition
- Study LCM / Consistency Models for 1-step generation
- Apply evaluation metrics to a real T2I model comparison